<a href="https://colab.research.google.com/github/leejunho12316/HonGongMachine/blob/main/RAG_%2B_Function_Calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain langchain-openai langchain-community langchain-text-splitters tiktoken chromadb pymupdf gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7

In [50]:
import os

os.environ['OPENAI_API_KEY'] = ""

#파일 별 vectorDB 적재 후 Retriever 생성 후 Tool로 지정

In [5]:
from typing import List
import requests

# PDF 파일 다운로드 함수
def download_pdfs(urls: List[str]) -> None:
    """URL 목록에서 PDF 파일을 다운로드합니다."""
    for url in urls:
        filename = url.split("/")[-1]
        response = requests.get(url)
        with open(filename, "wb") as f:
            f.write(response.content)
        print(f"{filename} 다운로드 완료")

# PDF 파일 다운로드
urls = [
    "https://raw.githubusercontent.com/llama-index-tutorial/llama-index-tutorial/main/ch06/ict_japan_2024.pdf",
    "https://raw.githubusercontent.com/llama-index-tutorial/llama-index-tutorial/main/ch06/ict_usa_2024.pdf"
]

download_pdfs(urls)

ict_japan_2024.pdf 다운로드 완료
ict_usa_2024.pdf 다운로드 완료


In [7]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# PDF에서 벡터 검색기(retriever) 생성 함수
def create_pdf_retriever(
    pdf_path: str,
    persist_directory: str,
    embedding_model: OpenAIEmbeddings,
    chunk_size: int = 512,
    chunk_overlap: int = 0
) -> Chroma.as_retriever:
    """PDF 문서를 처리하여 벡터 검색기를 생성합니다."""

    # PDF 파일 로드
    loader = PyMuPDFLoader(pdf_path)
    data = loader.load()

    # 텍스트 청킹
    text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    doc_splits = text_splitter.split_documents(data)

    # 벡터 스토어 생성
    vectorstore = Chroma.from_documents(
        persist_directory=persist_directory,
        documents=doc_splits,
        embedding=embedding_model,
    )

    return vectorstore.as_retriever()

In [8]:
# 임베딩 모델 초기화
embd = OpenAIEmbeddings()

# 각 국가별 검색기 생성
retriever_japan = create_pdf_retriever(
    pdf_path="ict_japan_2024.pdf",
    persist_directory="db_ict_policy_japan_2024",
    embedding_model=embd
)

retriever_usa = create_pdf_retriever(
    pdf_path="ict_usa_2024.pdf",
    persist_directory="db_ict_policy_usa_2024",
    embedding_model=embd
)

#도구 설정

In [24]:
from langchain_core.tools import tool

#LangChain 도구 지정 방법
#1. BaseTool 상속받는 클래스 지정 (지금까지 한 것)
#2. LangChain Runnables 지정 (아직 안배운거)
#3. 데코레이터로 도구 지정 방법 (밑에거)

@tool
def search_japan_ict(query: str) -> str:
    """일본의 ICT 시장동향 정보를 검색합니다. 일본 ICT와 관련된 질문에 사용하세요."""
    docs = retriever_japan.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

@tool
def search_usa_ict(query: str) -> str:
    """미국의 ICT 시장동향 정보를 검색합니다. 미국 ICT와 관련된 질문에 사용하세요."""
    docs = retriever_usa.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

In [25]:
print(type(search_japan_ict))
print(search_japan_ict.name)
print(search_japan_ict.description)
print()
print(search_japan_ict.args_schema.model_json_schema())

<class 'langchain_core.tools.structured.StructuredTool'>
search_japan_ict
일본의 ICT 시장동향 정보를 검색합니다. 일본 ICT와 관련된 질문에 사용하세요.

{'description': '일본의 ICT 시장동향 정보를 검색합니다. 일본 ICT와 관련된 질문에 사용하세요.', 'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'search_japan_ict', 'type': 'object'}


#LLM 생성

In [28]:
from langchain.agents import create_agent

llm = ChatOpenAI(model = 'gpt-4o', temperature = 0)

tools = [search_japan_ict, search_usa_ict]

# 시스템 프롬프트 정의
system_prompt = """당신은 ICT 정책 전문가입니다. 사용자의 질문에 최선을 다해 답변하세요.

## 지시사항
1. 함수를 호출하여 일본과 미국의 ICT 정책에 대한 정보를 검색하고 분석할 수 있습니다.
2. 일본과 미국 ICT와 연관없는 질문은 함수를 호출하지 않습니다.
3. 일본과 미국 ICT와 연관있는 질문은 반드시 함수를 호출하여 해결해야 합니다. 이는 가장 중요한 지시사항입니다. 반드시 준수하세요.
4. 일본과 미국 ICT와 연관있는 질문은 반드시 함수를 호출하여 해결해야 합니다. 당신이 자의적으로 판단해서 호출하지 않는 행동은 지양해야 합니다.
5. 전혀 연관없는 질문이 아니라면 반드시 함수를 호출하여 답변하십시오.

이전 대화 내용을 기억하고 참조하여 일관된 답변을 제공하세요."""
agent = create_agent(model = llm, tools = tools, system_prompt = system_prompt)

##1

In [31]:
result = agent.invoke({'messages':'미국의 기관 협력 사례'})

In [43]:
for r in result['messages']:
  print(type(r),"||", r)
  print()

<class 'langchain_core.messages.human.HumanMessage'> || content='미국의 기관 협력 사례' additional_kwargs={} response_metadata={} id='3afad015-9c11-4632-b000-f0d678cc7b38'

<class 'langchain_core.messages.ai.AIMessage'> || content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 280, 'total_tokens': 305, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_7c4b976237', 'id': 'chatcmpl-DIYFBuerLYrRoflVyyk8K3O942abs', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019ce1be-de2b-7100-b8d9-6162cbc733b4-0' tool_calls=[{'name': 'search_usa_ict', 'args': {'query': '미국의 ICT 분야 기관 협력 사례'}, 'id': 'call_jRFFYdVZzh2arkmt1At5Iuqv', 'type': 'tool_call'}] invalid_to

In [46]:
print(result['messages'][-1].content)

미국의 ICT 분야에서의 기관 협력 사례는 여러 가지가 있습니다. 주요 사례로는 다음과 같은 것들이 있습니다:

1. **한국-미국 ICT 협력**: 과학기술정보통신부와 미국 기관 간의 협력이 주목받고 있습니다. 이는 한-미 FTA 체결 및 개정과 관련된 협력의 일환으로, 양국 간의 기술 교류와 협력을 강화하는 데 기여하고 있습니다.

2. **미국-싱가포르 기술 파트너십**: 미국과 싱가포르는 전략적 기술 파트너십을 강화하고 있으며, 이는 연구, 혁신 및 상업적 관계를 강화하여 과학적 지식의 국경을 확장하고 번영을 촉진하는 것을 목표로 하고 있습니다.

3. **미국-일본 양자컴퓨팅 협력**: 미국과 일본은 양자컴퓨팅 및 첨단 기술 협력을 강화하고 있으며, 이는 반도체, 인공지능 분야에서도 협력의 중요성을 강조하고 있습니다. 이는 미-중 기술 경쟁에 대응하기 위한 전략적 노력으로 평가됩니다.

이러한 협력 사례들은 미국이 다양한 국가와의 기술 교류를 통해 ICT 분야에서의 발전을 도모하고 있음을 보여줍니다.


##2

In [47]:
result = agent.invoke({'messages':'일본의 ICT 발전 현황'})

In [49]:
for r in result['messages']:
  print(type(r),"||", r)
  print()
print(result['messages'][-1].content)

<class 'langchain_core.messages.human.HumanMessage'> || content='일본의 ICT 발전 현황' additional_kwargs={} response_metadata={} id='1b228d0c-39ab-4800-8c4a-900420a6734d'

<class 'langchain_core.messages.ai.AIMessage'> || content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 280, 'total_tokens': 303, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_97cd4ae281', 'id': 'chatcmpl-DIYOibIN1wuut5aTUKKVGJlB8QRHv', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019ce1c7-e4f5-70f0-b3b0-1b76b135bf18-0' tool_calls=[{'name': 'search_japan_ict', 'args': {'query': '일본의 ICT 발전 현황'}, 'id': 'call_uXpYnokmJbyeexajaCX6tEIp', 'type': 'tool_call'}] invalid_tool_